# Portfolio analysis with the saved risk model

Loads the model saved by `style_factors.ipynb` (the `model/` directory) — no raw data,
loads in milliseconds. Weights are fractions of NAV: net ≠ 1 and short weights are fine;
tickers outside the model universe are warned about and ignored.

In [1]:
from risk_model import RiskModel

m = RiskModel.load('model')
m.meta

{'fit_date': '2026-08-07',
 'window': ['2018-11-20', '2026-08-07'],
 'n_days': 1937,
 'n_names': 503,
 'mean_xsec_r2': 0.3759,
 'style_factors': ['Beta', 'Momentum', 'Size', 'ResVol'],
 'annualization': 252,
 'notes': 'exposures z-scored to EW mean 0 / std 1, lagged 1 day; Size = log 63d median dollar volume (size/liquidity proxy)'}

## Example 1 — concentrated mega-cap tech book

High factor vol (all one sector, high-beta names) *and* meaningful specific risk
(only five names — diversification hasn't killed the idiosyncratic term).

In [2]:
tech = {'AAPL': 0.20, 'MSFT': 0.20, 'NVDA': 0.20, 'GOOGL': 0.20, 'AMZN': 0.20}
d = m.report(tech)

net +1.00  gross 1.00   (model fit: 2026-08-07)
total vol    :  24.92%
  factor     :  22.55%   (81.9% of variance)
  specific   :  10.61%   (18.1% of variance)
style exposures (z-units): Beta -0.28  Momentum +0.08  Size +3.07  ResVol +0.20

top factor variance contributors:
Information Technology    0.418
Size                      0.203
Consumer Discretionary    0.118
Communication Services    0.116
ResVol                    0.011
Momentum                  0.001

top specific variance contributors:
NVDA     0.057
AMZN     0.043
AAPL     0.032
GOOGL    0.031
MSFT     0.020


## Example 2 — dollar-neutral long/short

Long integrated energy, short regulated utilities, net exposure zero. Sector risk
dominates by construction; note the style tilts the book picks up along the way.

In [3]:
ls = {'XOM': 0.33, 'CVX': 0.33, 'COP': 0.34,
      'NEE': -0.33, 'DUK': -0.33, 'SO': -0.34}
d = m.report(ls)

net -0.00  gross 2.00   (model fit: 2026-08-07)
total vol    :  29.79%
  factor     :  26.91%   (81.6% of variance)
  specific   :  12.78%   (18.4% of variance)
style exposures (z-units): Beta -0.53  Momentum +0.18  Size +0.83  ResVol +0.08

top factor variance contributors:
Energy       0.561
Utilities    0.238
Beta         0.014
Size         0.001
Momentum     0.001
ResVol       0.001

top specific variance contributors:
NEE    0.045
COP    0.040
CVX    0.033
XOM    0.032
SO     0.020
DUK    0.014


## Other entry points

- `m.exposures(w)` — the 15 factor exposures of any weight vector
- `m.decompose(w)` — the full decomposition as a dict (for further computation)
- `m.covariance([...])` — model-implied asset covariance block for any symbols
- `m.factor_returns` — daily factor return history, for attribution or plots

In [4]:
m.covariance(['AAPL', 'MSFT', 'XOM', 'NEE']).round(4)

,AAPL,MSFT,XOM,NEE
AAPL,0.1054,0.0505,0.0128,0.0212
MSFT,0.0505,0.0805,0.0125,0.0204
XOM,0.0128,0.0125,0.0999,0.0190
NEE,0.0212,0.0204,0.0190,0.0775
